In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 0.6
opt = SplineOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate,
                lam_rob=0.1, lam_leak=1, lam_dark=1);
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 100; verbose=false, pre_threshold=0.01) # default verbosity is true

obj = 0.024544291100757176
obj = 0.0012616502584855312
obj = 0.0012528045404138777
obj = 0.0007787904293963714
Round 20 done
obj = 0.0007325159363588217
Round 40 done
Round 60 done
Round 80 done
obj = 0.0007303938736164321
Round 100 done
165.429912 seconds (396.29 k allocations: 20.584 MiB, 0.14% compilation time)


(0.0007303938736164321, [-32.02842577956072, 28.643175244944754, 62.83185307179586, 62.83185307179586, 52.074146628153756, 8.846667387194433, -4.299244144382874, -1.1345365336044655, -3.0152543634153908, 12.382851971231874  …  20.403765039054356, 25.728948499352022, 36.51392888797581, 21.934440362379014, 31.681981004185737, 1.96629350025335, 5.58501393161755, -7.192321306219068, -7.132979175595295, -10.966353509306227])

In [6]:
# More tries to refine the result.
for _ in 1:25
    best_obj, best_args = @time opt_n!(opt, 40; pre_threshold=0.01,
                                       verbose=false, best_obj=best_obj, best_args=best_args)
    if best_obj < 0.00075
        break
    end
end

Round 20 done
Round 40 done
 67.842096 seconds (3.61 k allocations: 175.797 KiB, 0.01% compilation time)


In [7]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

[0.0, -0.021352283853040478, -0.04125316930799134, -0.05970334473369047, -0.07670418686781365, -0.09225776081687467, -0.10636682005622501, -0.1190348064300542, -0.13026585015138964, -0.1400647698020966, -0.14843707233287817, -0.15538895306327577, -0.16092729568166791, -0.1650596722452717, -0.16779434318014208, -0.1691402572811715, -0.1691070517120906, -0.16770505200546781, -0.16494527206270956, -0.16083941415406003, -0.15539986891860147, -0.14863971536425388, -0.14057272086777525, -0.13121334117476147, -0.12057672039964629, -0.10867869102570138, -0.09553577390503623, -0.08116517825859826, -0.06558480167617291, -0.0488132301163834, -0.03086973790669082, -0.011774287743394335, 0.008452503777868874, 0.029789504040122042, 0.05221509887454877, 0.07570719256049256, 0.1002432078254574, 0.12580008584510746, 0.15235428624326686, 0.1798817870919203, 0.20835808491121238, 0.23775819466944817, 0.26805664978309274, 0.29922750211677107, 0.3312443219832693, 0.3640801981435331, 0.3977077378066681, 0.43

In [8]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "Omega"=>Ω))